In [1]:
import yaml
import torch
import pytorch_lightning as pl
from tqdm import tqdm
import torch.nn as nn
import numpy as np 

In [2]:
import sys 
sys.path.append('../')
from src.attn_tracking_lightning import AttentionalTrackingModule
from corpus.jsinV3AttnTrackingValidation import jsinV3_attn_tracking_validation
from corpus.jsinV3_attn_multi_talker_w_audioset import jsinV3_attn_multi_talker_w_audioset
import src.audio_transforms as at


In [3]:
path = 'config/control_attn/dual_channel_attn.yaml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [18]:

audio_config = config['data']['audio']

audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), # set target and distractor to same level
            at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
            at.UnsqueezeAudio(dim=0),
            # at.AudioToAudioRepresentation(**audio_config),
])

bg_combine_transforms = at.AudioCompose([
        at.AudioToTensor(),
        at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), # set target and distractor to same level
        at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
        at.UnsqueezeAudio(dim=0),
    ])


cochgram_transforms = at.AudioCompose([
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config),

])

center_crop=False
binaural=False
using FIR cochleagram


In [15]:
model.audio_transforms

AudioCompose(
    AudioToTensor()
    CombineWithRandomDBSNR()
    RMSNormalizeForegroundAndBackground()
    UnsqueezeAudio()
)

In [17]:
model.bg_combine_transforms

AudioCompose(
    AudioToTensor()
    CombineWithRandomDBSNR()
    RMSNormalizeForegroundAndBackground()
    UnsqueezeAudio()
)

In [20]:

config['layernorm_first'] = True
config['data']['corpus']['n_talkers'] = 1 

model = AttentionalTrackingModule(config)

dataset = jsinV3_attn_multi_talker_w_audioset(**config['data']['corpus'],
                                          mode='val',
                                          with_audioset = False,
                                          transform=[audio_transforms,bg_combine_transforms],
                                          demo=True)

dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=1,
            num_workers=0
        )

ln_first
Using dual channel architecture
center_crop=False
binaural=False
using FIR cochleagram


In [21]:
ckpt_path = 'attn_cue_models/dual_channel_attn/checkpoints/epoch=1-step=16349-v1.ckpt' 

ckpt = torch.load(ckpt_path, map_location=torch.device('cpu'))

model.load_state_dict(ckpt['state_dict'])


<All keys matched successfully>

In [22]:

activations = {}


def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook



<generator object Module.modules at 0x2b340f099190>

In [50]:
{name:module for name, module in model.model.named_children()}['0']

AttnAudioInputRepresentation(
  (full_rep): AudioToAudioRepresentation(
    (rep): AudioToCochlearRep(
      (downsampling_op): Resample()
      (Cochleagram): TimeDomainCochleagram(
        (compute_rep): ComputeSubbands()
        (downsampling): Resample()
      )
    )
    (compression): ClippedGradPower(
      (compression_function): ClippedGradPowerCompression()
    )
  )
)

In [27]:
n_activations = 100
# set hooks for time_average layers & make SigmoidLayers


modules = {name:module for name, module in model.model.named_children()}

coch_transform = modules["0"]
model_layers = modules["0"]

cue_reps = {}
fg_reps = {}
bg_reps = {}
mixture_reps = {}

for name, module in conv_modules.items():
    print(name)
    if 'relufc' in name:
        module.register_forward_hook(get_activation(name)) # [2] is relu 
    else:
        module[2].register_forward_hook(get_activation(name)) # [2] is relu 
        
    cue_reps[name] = []
    fg_reps[name] = []
    bg_reps[name] = []
    mixture_reps[name] = []



NameError: name 'module' is not defined

In [14]:
model = model.eval().cuda()
# model = model.eval()


# add cochleagram to dicts 

mixture_reps['cochleagram'] = []
fg_reps['cochleagram'] = []
bg_reps['cochleagram'] = []
cue_reps['cochleagram'] = []

with torch.no_grad():
    for ix, batch in tqdm(enumerate(dataloader),  total = n_activations):
        foreground, background, mixture, fg_cue, fg_target = batch
        
        # convert to cochleagrams
        foreground, _ = cochgram_transforms(foreground, None)
        background, _ = cochgram_transforms(background, None)
        foreground = foreground.squeeze(0)
        background = background.squeeze(0)
        fg_cue = fg_cue.squeeze(0).squeeze(0)
        print(fg_cue.shape)


        # save inputs
        cue_reps['cochleagram'].append(fg_cue.view(1,-1))
        mixture_reps['cochleagram'].append(mixture.view(1,-1))
        fg_reps['cochleagram'].append(foreground.view(1,-1))
        bg_reps['cochleagram'].append(background.view(1,-1))

        # send to device
        foreground, background, mixture = foreground.cuda(), background.cuda(), mixture.cuda()
        fg_cue =  fg_cue.cuda()
        
        # run cue, cue for shared var 
        model(fg_cue, fg_cue)
        for layer in cue_reps.keys():
            if layer == 'cochleagram':
                continue 
            cue_reps[layer].append(activations[layer].view(1,-1).cpu())
                
        # run fg


        # run mixture
        model(fg_cue, mixture)
            
        for layer in mixture_reps.keys():
            if layer == 'cochleagram':
                continue 
            rep = activations[layer].view(1,-1).cpu()
            # account for var 
            rep = rep - cue_reps[layer]
            mixture_reps[layer].append(rep)
                
        # run fg
        model(fg_cue, foreground)
            
        for layer in fg_reps.keys():
            if layer == 'cochleagram':
                continue 
            rep = activations[layer].view(1,-1).cpu()
            # account for var
            rep = rep - cue_reps[layer]
            fg_reps[layer].append(rep)
                
        # run bg
        model(fg_cue, background)
            
        for layer in bg_reps.keys():
            if layer == 'cochleagram':
                continue 
            rep = activations[layer].view(1,-1).cpu()
            # account for var
            rep = rep - cue_reps[layer]
            bg_reps[layer].append(rep)
        
        if ix % 10 == 0:
            print(f"on step {ix}")
        if ix == n_activations-1:
            break 
            

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [01:56<?, ?it/s]

torch.Size([1, 40, 16000])


RuntimeError: Given normalized_shape=[2, 40, 16000], expected input with shape [*, 2, 40, 16000], but got input of size[1, 2, 40, 256000]

In [13]:
mixture

torch.Size([1, 1, 40, 16000])

In [ ]:
cue_reps = {layer:torch.concat(acts,axis=0).numpy().astype('float16') for layer,acts in cue_reps.items()}

mixture_reps = {layer:torch.concat(acts,axis=0).numpy().astype('float16')
                     for layer,acts in mixture_reps.items()}

fg_reps = {layer:torch.concat(acts,axis=0).numpy().astype('float16')
                for layer,acts in fg_reps.items()}

bg_reps = {layer:torch.concat(acts,axis=0).numpy().astype('float16')
                for layer,acts in bg_reps.items()}


In [12]:
import pickle

out_dict = dict(mixture_reps=mixture_reps, fg_reps=fg_reps, bg_reps=bg_reps)
out_name = '../attn_cue_models/dual_channel_attn/model_output_reps_0dB.pkl'


with open(out_name ,'wb') as f:
    pickle.dump(out_dict, f, protocol=pickle.HIGHEST_PROTOCOL)
    
out_name = '../attn_cue_models/dual_channel_attn/model_output_reps_0dB.npz'

np.savez(out_name, mixture_reps=mixture_reps, fg_reps=fg_reps, bg_reps=bg_reps)